#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, regexp_replace, substring, length, startswith
from pyspark.sql.window import Window
import uuid
from datetime import datetime

In [0]:
# crm_prd_info_raw time (starts here...)

start_time = datetime.now()

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.crm_prd_info_raw")

# Data Transformations

## Replace dashes with underscores

In [0]:
# erp table for categories contain '_'

df = df.withColumn("prd_key", regexp_replace(col("prd_key"), "-", "_"))

## Derive Product Cat ID and remove from prd_key column

In [0]:
df = df.withColumn("cat_id", substring(col("prd_key"), 1, 5))
df = df.withColumn("prd_key", substring(col("prd_key"), 7, length(col("prd_key"))))

# inserting Cat ID at the start
cols = df.columns
cols.insert(1, cols.pop(cols.index("cat_id")))
df = df.select(cols)

## Normalization

In [0]:
# mapping to the meaningful name

df = (
        df
        .withColumn(
                "prd_line",
                F.when(F.upper(trim(col("prd_line"))) == 'R', 'Road')
                 .when(F.upper(trim(col("prd_line"))) == 'M', 'Mountain')
                 .when(F.upper(trim(col("prd_line"))) == 'S', 'Other Sales')
                 .when(F.upper(trim(col("prd_line"))) == 'T', 'Touring')
                 .otherwise('n/a')
                )
     )

## Handle Product Cost

In [0]:
df = df.withColumn("prd_cost", F.when(col("prd_cost").isNull(), 0).otherwise(col("prd_cost")))

##Invalid Dates

In [0]:
window = Window.partitionBy('prd_key').orderBy('prd_start_dt')

df = df.withColumn(
    'prd_end_dt', 
    F.lead(col("prd_start_dt")-1).over(window)
)

## Renaming Colmns

In [0]:
rename_map = {
    "prd_id"       : "product_id",
    "cat_id"       : "category_id",
    "prd_key"      : "product_key",
    "prd_nm"       : "product_name",
    "prd_cost"     : "product_cost",
    "prd_line"     : "product_line",
    "prd_start_dt" : "start_date",
    "prd_end_dt"   : "end_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)
# df.columns

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable('salesdatalakehouse.silver.crm_prd_info')

#crm_prd_info Logging

In [0]:

# crm_prd_info_raw time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Silver'
schema_name = 'silver'
table_name = 'crm_prd_info'
row_inserted = spark.sql(''' SELECT COUNT(*) FROM salesdatalakehouse.silver.crm_prd_info''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ crm_prd_info info Logged with run {run_id}")